In [68]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import deque
import random
import logging
from game_logic import Direction, GameState, GameEngine

In [69]:
# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [70]:
# Define the DQN Network
input_size = 12  # Adjust based on state size
hidden_size = 64
output_size = len(Direction)  # Number of possible actions

policy_net = nn.Sequential(
    nn.Linear(input_size, hidden_size),
    nn.ReLU(),
    nn.Linear(hidden_size, hidden_size),
    nn.ReLU(),
    nn.Linear(hidden_size, output_size)
)

target_net = nn.Sequential(
    nn.Linear(input_size, hidden_size),
    nn.ReLU(),
    nn.Linear(hidden_size, hidden_size),
    nn.ReLU(),
    nn.Linear(hidden_size, output_size)
)

target_net.load_state_dict(policy_net.state_dict())

<All keys matched successfully>

In [71]:
# Training parameters
device = torch.device("cpu")
optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
memory = deque(maxlen=10000)
batch_size = 64
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995
target_update = 10
steps_done = 0

In [72]:
logger.info("ML Agent initialized")

2025-02-21 00:59:41,022 - __main__ - INFO - ML Agent initialized


In [73]:
# Function to convert game state to input features
def get_state(game_state, snake_id):
    snake = game_state.snakes[snake_id]
    head = snake.body[0]
    
    nearest_token_dist = float('inf')
    nearest_token_dir = (0, 0)
    
    for token in game_state.tokens:
        if token.active:
            dx = token.position[0] - head[0]
            dy = token.position[1] - head[1]
            dist = (dx**2 + dy**2)**0.5
            
            if dist < nearest_token_dist:
                nearest_token_dist = dist
                nearest_token_dir = (dx/dist if dist > 0 else 0, dy/dist if dist > 0 else 0)
    
    danger = [0, 0, 0, 0]  # [up, right, down, left]
    for i, direction in enumerate([Direction.UP, Direction.RIGHT, Direction.DOWN, Direction.LEFT]):
        next_pos = (head[0] + direction.value[0], head[1] + direction.value[1])
        
        if (next_pos[0] < 0 or next_pos[0] >= game_state.board_size[0] or
            next_pos[1] < 0 or next_pos[1] >= game_state.board_size[1]):
            danger[i] = 1
        
        if next_pos in snake.body[1:]:
            danger[i] = 1
    
    state = [
        snake.direction == Direction.UP.value,
        snake.direction == Direction.RIGHT.value,
        snake.direction == Direction.DOWN.value,
        snake.direction == Direction.LEFT.value,
        *danger,
        *nearest_token_dir
    ]
    
    return np.array(state, dtype=np.float32)

In [74]:
# Function to select an action using epsilon-greedy policy
def select_action(state):
    global epsilon
    if random.random() < epsilon:
        action_idx = random.randrange(output_size)
    else:
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            action_values = policy_net(state_tensor)
            action_idx = action_values.max(1)[1].item()
    
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    return action_idx, list(Direction)[action_idx]

In [75]:
# Function to store experience in replay memory
def store_experience(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

In [76]:
# Function to train the agent
def train():
    global steps_done
    if len(memory) < batch_size:
        return 0.0
    
    transitions = random.sample(memory, batch_size)
    batch = list(zip(*transitions))
    
    state_batch = torch.FloatTensor(batch[0]).to(device)
    action_batch = torch.LongTensor(batch[1]).to(device)
    reward_batch = torch.FloatTensor(batch[2]).to(device)
    next_state_batch = torch.FloatTensor(batch[3]).to(device)
    done_batch = torch.FloatTensor(batch[4]).to(device)
    
    current_q_values = policy_net(state_batch).gather(1, action_batch.unsqueeze(1))
    next_q_values = target_net(next_state_batch).max(1)[0].detach()
    expected_q_values = reward_batch + (1 - done_batch) * gamma * next_q_values
    
    loss = nn.MSELoss()(current_q_values.squeeze(), expected_q_values)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if steps_done % target_update == 0:
        target_net.load_state_dict(policy_net.state_dict())
    
    steps_done += 1
    return loss.item()

In [77]:
# Initialize game
game_engine = GameEngine()
player_id = game_engine.initialize_player("TestPlayer", ["frontend"])
snake_id = f"{player_id}_frontend"

In [78]:
# Run test episode
state = game_engine.get_state()
done = False
total_reward = 0

In [79]:
print("\nTesting ML Agent:")
for step in range(100):  # Run for 100 steps max
    agent_state = get_state(game_engine.state, snake_id)
    action_idx, direction = select_action(agent_state)
    
    game_engine.make_move(snake_id, direction.value)
    new_state = game_engine.update()
    
    snake = new_state.snakes[snake_id]
    reward = snake.score - total_reward
    done = not snake.is_alive
    
    print(f"\nStep {step + 1}:")
    print(f"Action: {direction.name}")
    print(f"Reward: {reward}")
    print(f"Snake Position: {snake.body[0]}")
    
    if done:
        break
        
    total_reward = snake.score


Testing ML Agent:

Step 1:
Action: RIGHT
Reward: 0
Snake Position: (27, 12)


In [80]:
print(f"\nEpisode finished after {step + 1} steps")
print(f"Total reward: {total_reward}")


Episode finished after 1 steps
Total reward: 0
